In [1]:
import copy

In [2]:
#Creating the Graph
G = Graph(12)

G.add_edges([(0,1),(1,2),(2,3),(1,3),(0,3)])

G.add_edges([(0,4), (1,5), (1,6), (1,7), (2,8), (3,9), (3,10), (3,11)])

In [3]:
#Labelling the half-edges
G.set_edge_label(0,4, (0,0))
G.set_edge_label(1,5,(1,0))
G.set_edge_label(1,6,(1,1))
G.set_edge_label(1,7,(1,2))
G.set_edge_label(2,8,(2,0))
G.set_edge_label(3,9,(3,0))
G.set_edge_label(3,10,(3,1))
G.set_edge_label(3,11,(3,2))

In [4]:
num_colors=6
#coloring = ((0,), (0, 1, 2), (0,), (0, 1, 2))

In [5]:
#!/usr/bin/env python

import itertools

def does_extend(G, coloring, num_colors):
    p = MixedIntegerLinearProgram()
    x= p.new_variable(binary=True)
    L= G.line_graph()
    for e in G.edges(): 
        p.add_constraint(p.sum(x[e,i] for i in range(num_colors)) == 1) 
    
    for e,f in L.edges(labels=False):
        for i in range(num_colors):
            p.add_constraint(x[e,i] + x[f,i] <=1)
    
    for e in G.edges():
        if e[2]: #is not none
            tuple_index,color_index = e[2]
            p.add_constraint(x[e, coloring[tuple_index][color_index] ] == 1)
    
    p.set_objective(0)
    try:
        p.solve()
        return True
    except sage.numerical.mip.MIPSolverException:
        #print(coloring)
        return False
    
def test_colorings():
    
    num_colors=6
    C=list(range(num_colors))  # list of colors, for convenience
    degrees=[1,3,1,3]  # this would come from the graph
    
    S=set()  # the set of colorings that can be modified to an extendible coloring
    U=set()
    for coloring in itertools.product(
        *(
          #(frozenset(x) for x in itertools.combinations(C,d))  # generator
          (tuple(x) for x in itertools.combinations(C,d))
          for d in degrees
         )
        ):
        if does_extend(G, coloring, num_colors):  #extend conditions
            S.add(coloring)
        else:
            U.add(coloring)
        #print(coloring)
    print(len(S), len(U))
    return S,U
    
S,U = test_colorings()

12120 2280


In [6]:
S1 = set() #list that does work after one change
U1 = set() #list that don't work

In [7]:
# recieves the two colors we are working with and the coloring
# returns the set of chains that are represented in the auxiliary graph
def perform_Kempe(coloring, i, j):
    aux_edges = set()
    # for trailing off Kempe Chains or loops in auxillary graph
    for k in range(len(coloring)):
        coloring_vertex = copy.deepcopy(list(coloring[k]))
        isChanged = 0
        for l in range(len(coloring_vertex)):
            test_coloring = copy.deepcopy(coloring)
            if coloring_vertex[l] == i:
                coloring_vertex[l] =j
                isChanged += 1
            else:
                if coloring_vertex[l] == j:
                    coloring_vertex[l]=i
                    isChanged += 1
        coloring_vertex.sort()
        if isChanged==1:
            test_coloring[k] = tuple(coloring_vertex)
            test_coloring = tuple(test_coloring)
            if test_coloring in S:
                S1.add(test_coloring)
            else:
                U1.add(test_coloring)
                aux_edges.add(tuple((k,)))
        if isChanged==2:
            test_coloring[k] = tuple(coloring_vertex)
            test_coloring = tuple(test_coloring)
            U1.add(test_coloring)
            aux_edges.add(tuple((k,)))

    #for Kempe chains between two vertices
    isKChanged = False
    isHChanged = False
    test_coloring = copy.deepcopy(coloring)
    for k in range(len(coloring)):
        for h in range(len(coloring)):
            if h>k:
                coloring_k = copy.deepcopy(list(coloring[k]))
                for l in range(len(coloring_k)):
                    if coloring_k[l] == i:
                        coloring_k[l] =j
                        isKChanged = True
                    else:
                        if coloring_k[l] == j:
                            coloring_k[l]=i
                            isKChanged = True
                coloring_h = copy.deepcopy(list(coloring[h]))
                for l in range(len(coloring_h)):
                    if coloring_h[l] == i:
                        coloring_h[l] =j
                        isHChanged = True
                    else:
                        if coloring_h[l] == j:
                            coloring_h[l]=i
                            isHChanged = True
                if (isKChanged and isHChanged) == True:
                    coloring_k.sort()
                    coloring_h.sort()
                    test_coloring[k] = tuple(coloring_k)
                    test_coloring[h] = tuple(coloring_h)
                    if tuple(test_coloring) in S:
                        S1.add(tuple(test_coloring))
                    else:
                        U1.add(tuple(test_coloring))
                        aux_edges.add(tuple((k,h)))
                isHChanged= False
                isKChanged= False
                test_coloring = copy.deepcopy(coloring)
    return aux_edges

In [11]:
def test_all_colors(coloring):
    CL = copy.deepcopy(list(coloring))
    for i in range(num_colors):
        for j in range(num_colors):
            if i < j:
                aux_edges=perform_Kempe(CL, i, j)
                print(f'{aux_edges} for {i} and {j}')
                
UL = list(U)
test_all_colors(UL[600])

set() for 0 and 1
{(0, 1)} for 0 and 2
{(0, 2)} for 0 and 3
set() for 0 and 4
{(0, 3)} for 0 and 5
{(1,), (3,), (1, 3)} for 1 and 2
set() for 1 and 3
{(1,), (3,), (1, 3)} for 1 and 4
{(1,), (3,), (1, 3)} for 1 and 5
{(1, 2)} for 2 and 3
{(1,), (3,), (1, 3)} for 2 and 4
{(1,), (3,), (1, 3)} for 2 and 5
set() for 3 and 4
{(2, 3)} for 3 and 5
{(1,), (3,), (1, 3)} for 4 and 5


In [32]:
((2,), (0, 1, 4), (3,), (1, 4, 5)) in U

True

In [54]:
# making the Mixed Integer Linear Program Constraints
p = MixedIntegerLinearProgram()
x= p.new_variable(binary=True)
L= G.line_graph()
for e in G.edges(): 
    p.add_constraint(p.sum(x[e,i] for i in range(num_colors)) == 1) 

for e,f in L.edges(labels=False):
    for i in range(num_colors):
        p.add_constraint(x[e,i] + x[f,i] <=1)

for e in G.edges():
    if e[2]: #is not none
        tuple_index,color_index = e[2]
        p.add_constraint(x[e, coloring[tuple_index][color_index] ] == 1)

p.set_objective(0)
p.show()        

Constraints:
  1.0 <= x_0 + x_1 + x_2 + x_3 + x_4 + x_5 <= 1.0
  1.0 <= x_6 + x_7 + x_8 + x_9 + x_10 + x_11 <= 1.0
  1.0 <= x_12 + x_13 + x_14 + x_15 + x_16 + x_17 <= 1.0
  1.0 <= x_18 + x_19 + x_20 + x_21 + x_22 + x_23 <= 1.0
  1.0 <= x_24 + x_25 + x_26 + x_27 + x_28 + x_29 <= 1.0
  1.0 <= x_30 + x_31 + x_32 + x_33 + x_34 + x_35 <= 1.0
  1.0 <= x_36 + x_37 + x_38 + x_39 + x_40 + x_41 <= 1.0
  1.0 <= x_42 + x_43 + x_44 + x_45 + x_46 + x_47 <= 1.0
  1.0 <= x_48 + x_49 + x_50 + x_51 + x_52 + x_53 <= 1.0
  1.0 <= x_54 + x_55 + x_56 + x_57 + x_58 + x_59 <= 1.0
  1.0 <= x_60 + x_61 + x_62 + x_63 + x_64 + x_65 <= 1.0
  1.0 <= x_66 + x_67 + x_68 + x_69 + x_70 + x_71 <= 1.0
  1.0 <= x_72 + x_73 + x_74 + x_75 + x_76 + x_77 <= 1.0
  x_0 + x_6 <= 1.0
  x_1 + x_7 <= 1.0
  x_2 + x_8 <= 1.0
  x_3 + x_9 <= 1.0
  x_4 + x_10 <= 1.0
  x_5 + x_11 <= 1.0
  x_0 + x_12 <= 1.0
  x_1 + x_13 <= 1.0
  x_2 + x_14 <= 1.0
  x_3 + x_15 <= 1.0
  x_4 + x_16 <= 1.0
  x_5 + x_17 <= 1.0
  x_0 + x_18 <= 1.0
  x_1 + x_19 

In [55]:
print('Objective Value: {}'.format(p.solve()))

Objective Value: 0.0


In [56]:
for e in G.edges():
    for i in range(num_colors):
        if p.get_values(x[e,i], convert = ZZ, tolerance = 1e-3) == 1:
            print(f'x[{e}, {i}] = 1')

x[(0, 1, None), 4] = 1
x[(0, 3, None), 5] = 1
x[(0, 4, (0, 0)), 0] = 1
x[(1, 2, None), 5] = 1
x[(1, 3, None), 3] = 1
x[(1, 5, (1, 0)), 0] = 1
x[(1, 6, (1, 1)), 1] = 1
x[(1, 7, (1, 2)), 2] = 1
x[(2, 3, None), 4] = 1
x[(2, 8, (2, 0)), 0] = 1
x[(3, 9, (3, 0)), 0] = 1
x[(3, 10, (3, 1)), 1] = 1
x[(3, 11, (3, 2)), 2] = 1
